# 02 Privacy muna: bawat row ay kapitbahay — notebook

Bago magbahagi si Maya ng kahit ano kay Doc Mon sa PESU, kailangan niyang gawing file na
nagpoprotekta sa mga tao rito ang hilaw na export ni Jun. Sa notebook na ito, ikaw ay:

1. maghahati ng mga column sa **direct identifiers**, **indirect identifiers** at
   **sensitive health information**;
2. mag-aalis ng direct identifiers — at mahuhuli si Tala na **nag-iwan ng pangalan** sa
   remarks;
3. makikita kung bakit **hindi anonymous ang simpleng hash ng pangalan**, at gagawa ng
   **salted** pseudonymous ID;
4. mag-**generalise** ng edad at petsa;
5. mag-**suppress ng maliliit na cell** sa table na ipa-publish mo;
6. susukat ng **re-identification risk** sa pamamagitan ng pagbilang kung ilang tao ang
   unique.

```{warning}
**PEKE ang mga pangalan, numero ng telepono at address sa `santa_luz_linelist_raw.csv`.**
Inimbento ang mga ito ng computer at sinadyang *mukhang* totoo para makapagpraktis ka.
Kathang-isip ang Santa Luz City at ang lahat ng tao rito. Hindi kailanman lumalabas ang
totoong line list sa mga protektadong computer ng opisina mo — at hindi kailanman
pumapasok sa notebook na ibinabahagi mo o sa AI chat.
```

Magandang gawi ito para sa training, hindi legal advice. Sa totoong trabaho, sundin ang
Data Protection Officer ng opisina mo at ang mga patakaran ng DOH na nalalapat sa iyo.

In [ ]:
# Google Colab setup -- skip this cell if running locally
import sys
import os
if 'google.colab' in sys.modules:
    !git clone https://github.com/ancientsky/python4epi.git /content/python4epi 2>/dev/null || true
    os.chdir('/content/python4epi')
    !pip install -q -e .

## 1. Buksan ang hilaw na export ni Jun

Ang unang linya ng file ay tala na nagsasabing synthetic ang datos, kaya nilalaktawan
natin ito gamit ang `skiprows=1`. Pinananatili ng `dtype=str` ang bawat cell bilang text,
eksakto kung paano na-type, at pinananatili ng `keep_default_na=False` ang mga blangkong
cell bilang empty text na `""` imbes na gawing "missing".

In [ ]:
import pandas as pd

raw = pd.read_csv(
    "data/synthetic/ph/santa_luz_linelist_raw.csv",
    skiprows=1, dtype=str, keep_default_na=False,
)
print(raw.shape)
raw.head(3)      # PEKENG pangalan, telepono at address

## 2. Aling mga column ang nakapagpapakilala sa tao?

- Ang **direct identifier** ay tumuturo sa iisang tao nang mag-isa: pangalan, numero ng
  telepono, street address, PhilHealth o ibang ID number.
- Ang **indirect identifier** (*quasi-identifier*) ay hindi nagpapangalan nang mag-isa,
  pero kaya ng ilan nito kapag pinagsama: ang barangay + edad + sex + petsa ng onset ay
  puwedeng iisang tao lang.
- Ang **sensitive health information** ay kung *tungkol saan* ang row: ang sakit, kung
  na-admit ang tao, kung namatay siya.
- Puwedeng maglaman ng alinman sa itaas ang **free text** (`Remarks`). Ituring itong
  direct identifier hangga't hindi mo pa nababasa ang bawat entry.

May bonus ang pagsulat ng mga role bilang dictionary: puwede nating **i-check** na may
role ang bawat column, para walang makalusot nang hindi nakaklasipika.

In [ ]:
roles = {
    "Patient Name": "direct identifier",
    "Contact No.": "direct identifier",
    "Address": "direct identifier",
    "Remarks": "free text (treat as direct)",
    "Reporting Unit": "indirect identifier",
    "Barangay": "indirect identifier",
    "Age": "indirect identifier",
    "Sex": "indirect identifier",
    "Date Reported": "indirect identifier",
    "Date of Onset": "indirect identifier",
    "Date Consulted": "indirect identifier",
    "Date Lab Confirmed": "indirect identifier",
    "Disease": "sensitive health information",
    "Classification": "sensitive health information",
    "Admitted": "sensitive health information",
    "Outcome": "sensitive health information",
    "Dengue Severity": "sensitive health information",
    "Waded in Floodwater": "sensitive health information",
    "Did Cleanup Work": "sensitive health information",
}

# Check: may role ang bawat column, at walang role na ibinigay sa column na hindi umiiral
assert set(roles) == set(raw.columns), set(raw.columns) ^ set(roles)
print(pd.Series(roles).value_counts())

## 3. Tanungin si Tala — gamit ang schema, hindi kailanman ang rows

**Hindi** nagpe-paste si Maya ng kahit anong row. Mga column lang ang inilalarawan niya:

> *May pandas DataFrame akong `raw` na may 19 na text column: Date Reported, Reporting
> Unit, Patient Name, Sex, Age, Contact No., Address, Barangay, Disease, Date of Onset,
> Date Consulted, Classification, Date Lab Confirmed, Admitted, Outcome,
> Waded in Floodwater, Did Cleanup Work, Dengue Severity, Remarks.
> Hindi ako puwedeng magbahagi ng kahit anong row: confidential na datos ito ng pasyente.
> Sumulat ng pandas code na mag-aalis ng direct identifiers para maibahagi ko ang table
> sa aming provincial epidemiologist.*

Sagot ni Tala:

In [ ]:
# Code ni Tala
direct_identifiers = ["Patient Name", "Contact No.", "Address"]
shared = raw.drop(columns=direct_identifiers)

print(shared.shape)
print(list(shared.columns))

## 4. Suriin: may pangalan pa ba sa file?

Tumakbo nang walang error ang code ni Tala, at wala na ang tatlong column. Pero ang
"walang error" ay hindi katulad ng "walang pangalan". Simpleng test: kunin ang bawat
**first name** mula sa column na kaaalis lang natin, at hanapin ito sa **bawat natitirang
cell**. Ang ibig sabihin ng search pattern na `\b(Ana|Dennis|...)\b` ay "isa sa mga
pangalang ito bilang buong salita".

In [ ]:
first_names = sorted(set(raw["Patient Name"].str.split().str[0]))
pattern = r"\b(?:" + "|".join(first_names) + r")\b"

hits = shared.apply(lambda column: column.str.contains(pattern)).sum()
print("Mga cell na may first name, ayon sa column:")
print(hits[hits > 0])

shared.loc[shared["Remarks"].str.contains(pattern), "Remarks"].head(5)

### Nagkamali si Tala

Apatnapung remarks pa ang may pangalan ng tao — *"anak/kapatid ni Dennis (kapitbahay)"*.
**Ibang tao** ang pinapangalanan ng remark na iyan, at kasama ang barangay, puwede itong
diretsong tumuro sa isang sambahayan. Delikado rin ang ibang remarks: pinapakitid ng
*"estudyante, Grade 5"* ang isang bata sa iisang klase.

Inalis ni Tala ang mga column na *tinatawag na* name, phone at address. Hindi niya nakita
ang laman ng `Remarks`, dahil (tama lang) hindi kailanman nagpakita si Maya sa kanya ng
kahit anong row. **Ang alam lang ng model ay ang sinasabi mo; trabaho mo ang mag-check ng
output.**

Ang ayos: alisin din ang `Remarks` (anumang may silbi rito ay dapat i-code muna sa loob ng
opisina bilang oo/hindi na column), at alisin ang `Reporting Unit` — isa o dalawang report
lang ang ipinadala ng ilang unit, kaya puwedeng tumuro sa isang tao ang pangalan ng unit.
Saka patakbuhin ulit ang parehong check: dapat **zero** ang makita nito.

In [ ]:
remove = ["Patient Name", "Contact No.", "Address", "Remarks", "Reporting Unit"]
shared = raw.drop(columns=remove)

hits = shared.apply(lambda column: column.str.contains(pattern)).sum()
assert hits.sum() == 0, "May pangalan pa sa table na ibabahagi!"
print(f"Wala nang first name sa {shared.shape[0]} rows x {shared.shape[1]} columns.")
print("Report bawat unit (kung bakit delikado ang pangalan ng unit):")
print(raw["Reporting Unit"].value_counts().tail(3))

## 5. Pseudonymous ID: bakit hindi sapat ang simpleng hash

Kailangan pa rin natin ng ID para masabi ni Doc Mon ang *"pakitingnan ulit ang case 17"*
at mahanap ito ni Maya. Ginagawa ng **hash** ang text na nakapirming halo-halong
character; laging pareho ang lumalabas sa parehong text. Maraming nag-aakalang nagiging
anonymous ang pangalan kapag na-hash. Hindi: ang sinumang may **listahan ng posibleng
pangalan** (listahan ng botante, class list, social media) ay puwedeng mag-hash ng bawat
pangalan dito at maghanap ng tugma. Sa cell sa ibaba, ginagampanan natin ang "attacker"
gamit ang gawa-gawang listahan ng limang pangalan.

In [ ]:
import hashlib


def plain_hash(name: str) -> str:
    return hashlib.sha256(name.encode()).hexdigest()[:12]


leaked_id = plain_hash("Juan P. Santos")          # ang laman ng file na walang salt
print("ID sa ibinahaging file:", leaked_id)

guess_list = ["Maria L. Reyes", "Jose A. Cruz", "Juan P. Santos", "Ana B. Garcia", "Mark D. Lim"]
for guess in guess_list:
    if plain_hash(guess) == leaked_id:
        print("Nahanap ng attacker ang tao:", guess)

**Magdagdag ng sikretong salt.** Inihahalo natin ang mahabang random na sikreto (ang
*salt*) sa hash gamit ang `hmac`. Kung walang salt, walang makauulit ng hashing, kaya
walang silbi ang guess list. Itinatago ng opisina ang salt sa nakakandadong file **sa loob
ng opisina** — hindi kailanman sa notebook, hindi kailanman sa AI chat, hindi kailanman
ipinapadala kasama ng datos. Kapag pareho ang salt linggo-linggo, pareho rin ang ID ng
parehong tao sa iba't ibang export.

Bago mag-hash, **nino-normalise** natin ang pangalan (malalaking titik, walang tuldok,
iisang space) para pareho ang ID ng `Juan P. Santos` at `JUAN P SANTOS`. Iyan mismo ang
tine-test ng mga check sa dulo, at na ibang ID ang lumalabas sa ibang salt.

Gumagawa ang `secrets.token_hex(16)` ng bagong random na salt tuwing tatakbo ito, kaya
**iba ang mga ID mo sa ID ng iba** — iyan nga ang punto.

In [ ]:
import hmac
import secrets

SALT = secrets.token_hex(16)     # demo lang: sa totoong trabaho, gumawa nang isang beses at ikandado


def pseudo_id(name: str, salt: str) -> str:
    key = " ".join(name.upper().replace(".", " ").split())   # i-normalise ang pangalan
    digest = hmac.new(salt.encode(), key.encode(), hashlib.sha256).hexdigest()
    return "P-" + digest[:10]


print(pseudo_id("Juan P. Santos", SALT))

# Checks
assert pseudo_id("Juan P. Santos", SALT) == pseudo_id("JUAN P SANTOS ", SALT)
assert pseudo_id("Juan P. Santos", SALT) != pseudo_id("Juan P. Santos", "another salt")
assert all(pseudo_id(g, SALT) != leaked_id for g in guess_list)

shared.insert(0, "pseudo_id", raw["Patient Name"].map(lambda n: pseudo_id(n, SALT)))
shared.head(3)

## 6. Generalise: age group at linggo

Malakas na indirect identifier ang eksaktong edad at eksaktong petsa. Para sa table na
lalabas ng opisina, pinapalawak natin ang mga ito: edad → age group, petsa → linggo.

Magulo pa ang mga hilaw na edad at petsa (`8 mos`, `24-Jul-26`, `46227`); nililinis ang
mga ito sa kabanata 03. Kaya mula rito, gagamitin natin ang **`santa_luz_linelist.csv`**,
ang malinis at de-identified na file na ginagawa ng kabanata 03 (sunod-sunod na ID na
`SLC-0001`…, walang pangalan, telepono, address, reporting unit o remarks).

Inilalagay ng `pd.cut` ang mga edad sa mga bin. Ang ibig sabihin ng `right=False` ay kasama
sa bawat bin ang kaliwang gilid nito, kaya sa `5-14` napupunta ang 5, hindi sa `0-4`. Ang
**check**: dapat tumugma ang mga grupo natin sa sariling `age_group` column ng file para
sa bawat row na may edad.

In [ ]:
df = pd.read_csv("data/synthetic/ph/santa_luz_linelist.csv")

labels = ["0-4", "5-14", "15-24", "25-44", "45-64", "65+"]
df["my_age_group"] = pd.cut(df["age"], bins=[0, 5, 15, 25, 45, 65, 200],
                            right=False, labels=labels)

has_age = df["age"].notna()
assert (df.loc[has_age, "my_age_group"].astype(str) == df.loc[has_age, "age_group"]).all()
print(f"Tugma ang age group sa lahat ng {has_age.sum()} kasong may edad "
      f"({(~has_age).sum()} ang walang edad).")

# Petsa -> ISO week ng onset (Lunes hanggang Linggo)
df["onset_week"] = pd.to_datetime(df["date_onset"]).dt.isocalendar().week
print(df[["date_onset", "onset_week"]].head(3))

## 7. Small-cell suppression bago mag-publish

Ang na-publish na table na may **1** ay puwedeng isang tao. *"Isang namatay sa dengue sa
Buhangin"* — balitang kayang lagyan ng pangalan ng buong barangay. Ang karaniwang
patakaran ay itago ang bilang na **1 hanggang 4** at ipakita ang `<5` (nananatiling zero
ang zero: wala itong ibinubunyag na tao).

Susubukan natin ito sa pinakasensitibong table: mga namatay ayon sa barangay at sakit.
Isa pang bitag: kung **iisa lang** ang nakatagong cell sa isang column, ang sinumang
nakaaalam ng total ng column (at kadalasang public ang city totals) ay kayang ibawas ang
mga nakikitang cell at makuha ito. Kaya **chine-check** din ng cell ang bawat column: kapag
iisa ang nakatagong cell, hindi puwedeng i-publish ang column na iyon ayon sa barangay.
Para sa ganoong column, city total lang ang ipa-publish natin at hindi isasama ang
barangay. Sensitibo pa rin kahit ang city total na 1: desisyon nina Doc Mon at ng Data
Protection Officer ng opisina kung ipa-publish ito, hindi ng code.

In [ ]:
def suppress(table: pd.DataFrame, threshold: int = 5) -> pd.DataFrame:
    if threshold < 1:
        raise ValueError("dapat hindi bababa sa 1 ang threshold")
    small = (table > 0) & (table < threshold)
    return table.astype(object).mask(small, f"<{threshold}")


deaths = df[df["outcome"] == "died"]
death_table = pd.crosstab(deaths["barangay"], deaths["disease"])
print(f"Namatay: {len(deaths)} sa {len(death_table)} barangay")

safe_table = suppress(death_table)
print(safe_table)

# Check: makukuha mula sa total ang column na may eksaktong ISANG nakatagong cell
hidden = (safe_table == "<5").sum()
recoverable = list(hidden[hidden == 1].index)
print("Mga column na iisa ang nakatagong cell:", recoverable)

publishable = safe_table.drop(columns=recoverable)
assert ((publishable == "<5").sum() != 1).all()
print(publishable)
for column in recoverable:
    print(f"{column}: city total lang ang i-publish ({death_table[column].sum()}), walang barangay.")

## 8. Re-identification check: ilang tao ang unique?

Delikado ang isang row kapag **unique** ito sa mga indirect identifier nito — walang ibang
row sa file na kapareho ng barangay, edad, sex, sakit, petsa ng onset… Ang ideya ng
**k-anonymity**: dapat may kahati ang bawat kombinasyon na hindi bababa sa *k* na tao
(halimbawa *k* = 5), kaya ang anumang row ay puwedeng isa sa hindi bababa sa lima.

Binibilang ng function, para sa bawat row, kung ilang row ang kapareho ng kombinasyon nito
(*k*), at iniuulat kung ilang row ang unique (*k* = 1). Susubukan natin ang apat na antas
ng detalye, mula eksakto hanggang malawak. Hindi kasama sa bawat bilang ang rows na may
kulang na value sa mga napiling column.

In [ ]:
import matplotlib.pyplot as plt


def uniqueness(data: pd.DataFrame, quasi: list[str]) -> dict:
    rows = data.dropna(subset=quasi)
    k = rows.groupby(quasi)[quasi[0]].transform("size")   # laki ng grupo ng bawat row
    return {"rows": len(rows), "unique (k=1)": int((k == 1).sum()), "k<5": int((k < 5).sum())}


levels = {
    "barangay + age + sex + disease + onset date": ["barangay", "age", "sex", "disease", "date_onset"],
    "barangay + age + sex + disease": ["barangay", "age", "sex", "disease"],
    "barangay + age group + sex + disease": ["barangay", "age_group", "sex", "disease"],
    "band + age group + sex + disease": ["band", "age_group", "sex", "disease"],
}
risk = pd.DataFrame({name: uniqueness(df, cols) for name, cols in levels.items()}).T
risk["% unique"] = (risk["unique (k=1)"] / risk["rows"] * 100).round(1)
print(risk)

ax = risk["% unique"].iloc[::-1].plot.barh(color="#CE1126", figsize=(8, 3.2))
ax.set_xlabel("% ng kasong unique sa mga column na ito (k = 1)")
ax.set_title("Mas malawak na column, mas kaunting unique na tao (Santa Luz, kathang-isip)")
ax.set_xlim(0, 100)
plt.tight_layout()
plt.show()

## Ikaw naman

1. **Mas maraming column, mas mataas ang risk.** Idagdag ang `"case_class"` sa
   `my_columns` sa ibaba at patakbuhin ang cell. Tumaas ba o bumaba ang bilang ng unique
   na rows? *Hint:* ang bawat dagdag na column ay puwede lang maghati ng mga grupo sa mas
   maliliit pa.
2. **Mas mahigpit na threshold.** May mga opisinang nagsu-suppress ng bilang na mas mababa
   sa 10. Patakbuhin ang `suppress(death_table, threshold=10)` (nasa cell na) at basahin
   ang resulta: gaano pa karami sa table ang may silbi?
3. **Sumulat ng ligtas na prompt.** Sa isang text cell, sumulat ng prompt na humihingi sa
   AI assistant mo ng code na bibilang ng kaso ng dengue ayon sa `barangay` at `age_group`
   at magsu-suppress ng mga cell na mas mababa sa 5. Pangalan lang ng column at dalawa o
   tatlong halatang gawa-gawang row ang gamitin (hal. barangay na `"Barangay X"`). Saka
   ikumpara ang code nito sa `suppress()` sa itaas.
   *Hint:* i-check na hindi nito ginagalaw ang mga zero.

In [ ]:
# --- Ikaw naman 1 ---
my_columns = ["band", "age_group", "sex", "disease"]      # <- idagdag ang "case_class"
print(uniqueness(df, my_columns))

# --- Ikaw naman 2 ---
suppress(death_table, threshold=10)

## Baon

- **Tingnan ang laman ng bawat column.** Hindi sapat na alisin ang mga column na
  *tinatawag na* name, phone at address: 40 ibang tao ang pinangalanan ng free-text
  remarks. Hanapin sa resulta ang mga inalis mo.
- **Mag-pseudonymise gamit ang sikreto.** Kayang baligtarin ng guess list ang simpleng hash
  ng pangalan; hindi kaya ng salted ID (nakakandado ang salt sa opisina) o ng sunod-sunod
  na ID na may nakakandadong key table.
- **Palawakin, i-suppress, saka sukatin.** Malaki ang ibinababa ng bahagi ng unique na tao
  dahil sa age group, linggo at `<5` na cell — at sasabihin ng uniqueness check kung sapat
  na ang ginawa mo.

Susunod: [kabanata 03, *Mula sa magulong line list tungo sa malinis na datos*](../03_clean_linelist.md).